# Doublet 2D layouts — 6h Blinatumomab, NALM-6 + healthy T (S006)

Pick 10 putative B–T conjugate doublets from S006, render the **A-pixel 2D layout** for each one twice:
- left:  CD19 / CD20 overlay (B side)
- right: CD3e / CD4 / CD8 overlay (T side)

PNA's precomputed layout is 3D (`pmds_3d`). We project to 2D via PCA on (x, y, z) so the doublet's long axis lies along the x-axis of the plot.

In [ ]:
import sys
sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from pixelator import read_pna

RESULTS_DIR     = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/results')
CACHE_DIR       = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

SAMPLE_ID = 'S006'  # 6h Blinatumomab, NALM-6 + healthy T
PXL_PATH  = RESULTS_DIR / SAMPLE_ID / 'layout' / 'layout' / f'{SAMPLE_ID}.layout.pxl'

B_MARKERS = ['CD19', 'CD20']
T_MARKERS = ['CD3e', 'CD4', 'CD8']

MIN_UMI = 25000
N_DOUBLETS = 10

plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 10, 'axes.labelsize': 9})

## 1. Load data

- `adata` — most recent annotated cache (cytovi compat), only used to confirm S006 is the 6h Blina HT+NALM6 sample.
- `pg` — raw PNA dataset for S006, needed because the annotated adata is filtered to `tau_type == 'normal'` and so does not contain any doublets.

In [ ]:
adata = sc.read_h5ad(ANNOTATED_CACHE)

ref = adata.obs.loc[adata.obs['sample'] == SAMPLE_ID, ['condition', 'time', 'cell_system']].drop_duplicates()
print(f'{SAMPLE_ID} in annotated adata →')
print(ref.to_string(index=False))
print(f'\nAnnotated cells from {SAMPLE_ID}: {(adata.obs["sample"] == SAMPLE_ID).sum()}')

In [ ]:
pg = read_pna(PXL_PATH)
adata_raw = pg.adata()
print(adata_raw)

print('\ntau_type counts:')
print(adata_raw.obs['tau_type'].value_counts().to_string())

## 2. Pick 10 doublets

A B–T conjugate doublet is a component carrying strong signal from both B markers (CD19, CD20) and T markers (CD3e, CD4, CD8). We rank cells with `n_umi ≥ MIN_UMI` by `min(B_frac, T_frac)` — the cells with highest joint signal are real conjugates, not B-only or T-only outliers.

`tau_type` in pixelator only has values `normal`/`high`/`low` (no `doublet`), so we don't filter on it here; we additionally print how the picked cells split across `tau_type` so you can see how many were flagged as aggregates (`high`).

In [ ]:
b_present = [m for m in B_MARKERS if m in adata_raw.var_names]
t_present = [m for m in T_MARKERS if m in adata_raw.var_names]
print(f'B markers present: {b_present}')
print(f'T markers present: {t_present}')

X = adata_raw.to_df()
umi = adata_raw.obs['n_umi'].astype(float)
b_frac = X[b_present].sum(axis=1) / umi
t_frac = X[t_present].sum(axis=1) / umi

adata_raw.obs['b_frac']  = b_frac
adata_raw.obs['t_frac']  = t_frac
adata_raw.obs['min_bt']  = np.minimum(b_frac, t_frac)

# Pure size filter, then rank by joint B/T signal. Cells with high min(B_frac, T_frac)
# are by definition B-T conjugates regardless of how pixelator's tau_type classified them.
size_mask = adata_raw.obs['n_umi'] >= MIN_UMI
print(f'\nCells with n_umi >= {MIN_UMI}: {size_mask.sum()} / {adata_raw.n_obs}')

top = (
    adata_raw.obs[size_mask]
    .sort_values('min_bt', ascending=False)
    .head(N_DOUBLETS)
)
print('\nPicked doublets (top by min(B_frac, T_frac)):')
print(top[['n_umi', 'tau', 'tau_type', 'b_frac', 't_frac', 'min_bt']].round(3).to_string())

print('\ntau_type breakdown of picked doublets:')
print(top['tau_type'].value_counts().to_string())

components = top.index.tolist()

## 3. Plot 2D layouts with B / T overlays

For each doublet:
- pull the precomputed 3D layout (`pmds_3d`) + per-node marker counts,
- keep A-pixels (markers live on the A side),
- project (x, y, z) → 2D via PCA (top 2 principal components — orients the doublet's long axis horizontally),
- left panel:  per-node sum of CD19+CD20,
- right panel: per-node sum of CD3e+CD4+CD8.

Color scales are per-panel (each cell gets its own vmin/vmax) so weak doublets stay legible.

In [ ]:
def get_2d_layout(pg, component):
    """Return DataFrame with columns (x2, y2, *markers) for A-pixels of one component."""
    df = pg.filter(components=[component]).precomputed_layouts().to_df()
    if 'pixel_type' in df.columns:
        df = df[df['pixel_type'] == 'A'].copy()
    elif 'node_type' in df.columns:
        df = df[df['node_type'] == 'A'].copy()
    coords = df[['x', 'y', 'z']].to_numpy()
    coords = coords - coords.mean(axis=0)
    _, _, Vt = np.linalg.svd(coords, full_matrices=False)
    proj = coords @ Vt.T[:, :2]
    df['x2'] = proj[:, 0]
    df['y2'] = proj[:, 1]
    return df


def plot_overlay(ax, df, markers, cmap, label):
    have = [m for m in markers if m in df.columns]
    sig = df[have].sum(axis=1) if have else pd.Series(0, index=df.index)
    order = sig.argsort().to_numpy()  # high-signal points on top
    sc = ax.scatter(
        df['x2'].to_numpy()[order],
        df['y2'].to_numpy()[order],
        c=sig.to_numpy()[order],
        cmap=cmap,
        s=4,
        alpha=0.85,
        linewidths=0,
        vmin=0,
        vmax=max(1, np.quantile(sig, 0.995)),
    )
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f'{label}: {"+".join(have)}', fontsize=9)
    return sc

In [ ]:
fig, axes = plt.subplots(N_DOUBLETS, 2, figsize=(8, 3.6 * N_DOUBLETS))
if N_DOUBLETS == 1:
    axes = axes[None, :]

for i, comp in enumerate(components):
    df = get_2d_layout(pg, comp)
    info = top.loc[comp]

    sc_b = plot_overlay(axes[i, 0], df, b_present, 'Reds',  'B')
    sc_t = plot_overlay(axes[i, 1], df, t_present, 'Blues', 'T')

    fig.colorbar(sc_b, ax=axes[i, 0], shrink=0.7, pad=0.02)
    fig.colorbar(sc_t, ax=axes[i, 1], shrink=0.7, pad=0.02)

    axes[i, 0].set_ylabel(
        f'{comp[:10]}\nUMI={int(info["n_umi"]):,}\nB={info["b_frac"]:.2f} T={info["t_frac"]:.2f}',
        rotation=0, ha='right', va='center', fontsize=8,
    )

fig.suptitle(f'{SAMPLE_ID} · 6h Blina · NALM-6 + healthy T — {N_DOUBLETS} doublets', fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

## 4. Combined B+T overlay (one panel per doublet)

Same projection, both signals on one plot. B side in red, T side in blue — a clean doublet has a red lobe and a blue lobe meeting at an interface.

In [ ]:
ncols = 5
nrows = int(np.ceil(N_DOUBLETS / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes)

for i, comp in enumerate(components):
    ax = axes[i // ncols, i % ncols]
    df = get_2d_layout(pg, comp)

    b_sig = df[b_present].sum(axis=1) if b_present else pd.Series(0, index=df.index)
    t_sig = df[t_present].sum(axis=1) if t_present else pd.Series(0, index=df.index)
    b_norm = (b_sig / max(1, np.quantile(b_sig, 0.995))).clip(0, 1).to_numpy()
    t_norm = (t_sig / max(1, np.quantile(t_sig, 0.995))).clip(0, 1).to_numpy()

    # White-background colour scheme:
    #   no signal       -> white      (1, 1, 1)
    #   pure B (CD19/20) -> red        (1, 0, 0)
    #   pure T (CD3/4/8) -> blue       (0, 0, 1)
    #   both             -> dark/black (interface)
    R = 1.0 - t_norm
    G = 1.0 - np.maximum(b_norm, t_norm)
    B = 1.0 - b_norm
    rgb = np.column_stack([R, G, B]).clip(0, 1)

    # Plot background (low-signal) nodes first, then signal nodes on top.
    sig = np.maximum(b_norm, t_norm)
    order = np.argsort(sig)
    ax.scatter(df['x2'].to_numpy()[order], df['y2'].to_numpy()[order],
               c=rgb[order], s=5, linewidths=0, alpha=1.0)

    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    info = top.loc[comp]
    ax.set_title(
        f'{comp[:10]}\nUMI={int(info["n_umi"]):,} · B={info["b_frac"]:.2f} T={info["t_frac"]:.2f}',
        fontsize=8,
    )

for j in range(N_DOUBLETS, nrows * ncols):
    axes[j // ncols, j % ncols].axis('off')

from matplotlib.patches import Patch
fig.legend(
    handles=[
        Patch(facecolor='red',  label=f'B side ({"+".join(b_present)})'),
        Patch(facecolor='blue', label=f'T side ({"+".join(t_present)})'),
        Patch(facecolor='lightgrey', edgecolor='grey', label='no signal'),
    ],
    loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle(f'{SAMPLE_ID} doublets — B (red) vs T (blue)', fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()